In [ ]:
import pandas as pd
from keras.callbacks import EarlyStopping
from keras.layers import Dense, Dropout
from keras.models import Sequential
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [ ]:
data = pd.read_csv("C:/Sarang/College/lp3/ml/Churn_Modelling.csv")

print("Dataset shape:", data.shape)
print(data.head())

In [ ]:
X = data.drop(["RowNumber", "CustomerId", "Surname", "Exited"], axis=1)

y = data["Exited"]


X = pd.get_dummies(
    X,
    columns=["Geography", "Gender"],
    drop_first=True
)

print("\nFeatures:")
print(X.columns)


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


print("\nTraining data:", X_train.shape)
print("Testing data:", X_test.shape)

In [ ]:
model = Sequential()

model.add(
    Dense(
        64,
        activation="relu",
        input_shape=(X_train.shape[1],)
    )
)

model.add(Dropout(0.30))

model.add(
    Dense(
        32,
        activation="relu"
    )
)

model.add(Dropout(0.20))

model.add(
    Dense(
        1,
        activation="sigmoid"
    )
)


model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


model.summary()

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)


history = model.fit(
    X_train,
    y_train,
    validation_split=0.20,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
y_pred_prob = model.predict(X_test)

y_pred = (y_pred_prob >= 0.5).astype(int).ravel()


accuracy = accuracy_score(
    y_test,
    y_pred
)

print("\nAccuracy:", accuracy)

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred
)

print("\nConfusion Matrix:")
print(cm)